In [ ]:
from google.colab import drive
drive.mount("/content/drive")

ValueError: Mountpoint must not already contain files

In [ ]:
import os
project_path = '/content/drive/MyDrive/deforestation_project'
os.makedirs(f'{project_path}/data/raw_imagery', exist_ok=True)
os.makedirs(f'{project_path}/data/labels', exist_ok=True)
os.makedirs(f'{project_path}/models/checkpoints', exist_ok=True)
os.makedirs(f'{project_path}/outputs/maps', exist_ok=True)
print("Folders created at:", project_path)

Folders created at: /content/drive/MyDrive/deforestation_project


In [ ]:

!pip install earthengine-api geemap rasterio segmentation-models-pytorch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 64.7 MB/s eta 0:00:00


In [ ]:
import ee
ee.Authenticate()
ee.Initialize(project='deforestation-detection-500907')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import rasterio
import geemap


In [ ]:
# Define an area
area = ee.Geometry.Rectangle([8.0, 4.8, 9.40, 6.9])

# Load Sentinel-2 image collection
collection = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(area) \
    .filterDate("2021-11-01", "2023-03-01") \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 10))

image = collection.median()

In [ ]:
sample = image.select(['B4', 'B3', 'B2']).reduceRegion(
    reducer=ee.Reducer.percentile([2, 98]),
    geometry=area,
    scale=100,
    maxPixels=1e9
).getInfo()

print(sample)

{'B2_p2': 48.207136087197675, 'B2_p98': 750.8589510665494, 'B3_p2': 305.9037719934832, 'B3_p98': 1038.8325754829077, 'B4_p2': 241.59442881637634, 'B4_p98': 1326.945458871024}


In [ ]:
Map = geemap.Map()
Map.centerObject(area, zoom=8)
Map.addLayer(
    image,
    {"bands" : ["B4" , "B3", "B2"], "min":50, "max": 1327, "gamma": 1.4},
    "Sentinel-2 True Colour"
)

Map

Map(center=[5.849776080704363, 8.700000000000172], controls=(WidgetControl(options=['position', 'transparent_b…

In [ ]:
hansen = ee.Image('UMD/hansen/global_forest_change_2025_v1_13')
Loss_year = hansen.select('lossyear').clip(area)



In [ ]:
hansen_assets = ee.data.listAssets('UMD/hansen')
for asset in hansen_assets['assets']:
    print(asset['id'])

UMD/hansen/MODIS_2015
UMD/hansen/MODIS_2015_1
UMD/hansen/global_forest_change_2013
UMD/hansen/global_forest_change_2014
UMD/hansen/global_forest_change_2015
UMD/hansen/global_forest_change_2015_v1_3
UMD/hansen/global_forest_change_2016_v1_4
UMD/hansen/global_forest_change_2017_v1_5
UMD/hansen/global_forest_change_2018_v1_6
UMD/hansen/global_forest_change_2019_v1_7
UMD/hansen/global_forest_change_2020_v1_8
UMD/hansen/global_forest_change_2021_v1_9
UMD/hansen/global_forest_change_2022_v1_10
UMD/hansen/global_forest_change_2023_v1_11
UMD/hansen/global_forest_change_2024_v1_12
UMD/hansen/global_forest_change_2025_v1_13


In [ ]:
loss_viz = {
    "min": 1,
    "max": 25,
    "palette": ["yellow", "orange", "red"]
}
# Load Hansen data as an ee.Image asset
hansen = ee.Image("UMD/hansen/global_forest_change_V1_13")
Loss_year = hansen.select("lossyear").clip(area)

Map = geemap.Map()
Map.centerObject(area, zoom=8)
Map.addLayer(
    image,
    {"bands" : ["B4" , "B3", "B2"], "min":50, "max": 1327, "gamma": 1.4},
    "Sentinel-2 True Colour"

)
Map.addLayer(Loss_year, loss_viz, "Forest Loss by Year")
Map



EEException: Image.load: Image asset 'UMD/hansen/global_forest_change_V1_13' not found (does not exist or caller does not have access).